**Task 2 (0.65p).** WordNet mini-challenge

(0.25p) Exploit WordNet’s hierarchy using relations such as hyponym/hypernym, antonyms,
meronyms, synonyms and definitions to collect words that are related to a user given word.

(0.25p) Build a Word Association Game using WordNet Similarity (any existing similarity method
from NLTK library), along with the relations identified above. The player is prompted with a word
and must come up with related words based on semantic similarity and relations using WordNet.
Offer rewards points based on how semantically similar the player's word is to the original word.
Provide feedback on the "closeness" of the player's word to the original.

(0.15p) Provide it in a user-friendly manner.

In [41]:
from nltk.corpus import wordnet as wn

In [42]:
def clean_word(word: str) -> str:
    return word.lower().replace('_', ' ')

relationship_types = ["synonyms", "antonyms", "hypernyms", "hyponyms", "meronyms"]

In [43]:
def get_related_words(word: str) -> None | dict[str, set[str] | list[str]]:
    synsets = wn.synsets(word)
    if len(synsets) == 0:
        return None

    target_clean = clean_word(word)

    results = {
        "definitions": [],
        "synonyms": set(),
        "antonyms": set(),
        "hypernyms": set(),
        "hyponyms": set(),
        "meronyms": set()
    }

    for synset in synsets:
        results["definitions"].append(f"({synset.pos()}) {synset.definition()}")

        for lemma in synset.lemmas():
            synonym_clean = clean_word(lemma.name())
            if synonym_clean != target_clean:
                results["synonyms"].add(synonym_clean)
            
            for antonym in lemma.antonyms():
                results["antonyms"].add(clean_word(antonym.name()))

        for hypernym_synset in synset.hypernyms():
            for lemma in hypernym_synset.lemmas():
                results["hypernyms"].add(clean_word(lemma.name()))

        for hyponym_synset in synset.hyponyms():
            for lemma in hyponym_synset.lemmas():
                results["hyponyms"].add(clean_word(lemma.name()))

        meronym_synsets = synset.part_meronyms() + synset.substance_meronyms() + synset.member_meronyms()
        for meronym_synset in meronym_synsets:
            for lemma in meronym_synset.lemmas():
                results["meronyms"].add(clean_word(lemma.name()))

    for relationship_type in relationship_types:
        results[relationship_type].discard(target_clean)

    return results

In [44]:
from pprint import pprint
pprint(get_related_words("car"))

{'antonyms': set(),
 'definitions': ['(n) a motor vehicle with four wheels; usually propelled by '
                 'an internal combustion engine',
                 '(n) a wheeled vehicle adapted to the rails of railroad',
                 '(n) the compartment that is suspended from an airship and '
                 'that carries personnel and the cargo and the power plant',
                 '(n) where passengers ride up and down',
                 '(n) a conveyance for passengers or freight on a cable '
                 'railway'],
 'hypernyms': {'automotive vehicle',
               'compartment',
               'motor vehicle',
               'wheeled vehicle'},
 'hyponyms': {'ambulance',
              'baggage car',
              'beach waggon',
              'beach wagon',
              'bus',
              'cab',
              'cabin car',
              'caboose',
              'carriage',
              'club car',
              'coach',
              'compact',
              'co

In [45]:
def evaluate_association(target_word: str, guess_word: str) -> tuple[float, list[str], bool]:
    target_synsets = wn.synsets(target_word)
    guess_synsets = wn.synsets(guess_word)

    max_similarity = 0.0
    for target_synset in target_synsets:
        for guess_synset in guess_synsets:
            # path_similarity returns a float [0, 1] or None
            path_similarity: float | None = target_synset.path_similarity(guess_synset)
            if path_similarity is not None and path_similarity > max_similarity:
                max_similarity = path_similarity

    relationships_found: list[str] = []
    related_words = get_related_words(target_word)
    clean_guess = clean_word(guess_word)

    if related_words is not None:
        for relationship_type in ["synonyms", "antonyms", "hypernyms", "hyponyms", "meronyms"]:
            if clean_guess in related_words[relationship_type]:
                relationships_found.append(relationship_type.capitalize())

    guess_is_in_dicitonary = len(guess_synsets) > 0

    return max_similarity, relationships_found, guess_is_in_dicitonary

In [46]:
def generate_feedback(similarity_score: float, relationships: list[str]) -> str:
    feedback_score = similarity_score + len(relationships) * 0.2
    if feedback_score >= 0.5:
        feedback = "Exceptional! Highly semantically related."
    elif feedback_score >= 0.33:
        feedback = "Great match! Strong relationship."
    elif feedback_score >= 0.20:
        feedback = "Decent connection, moderate similarity."
    elif feedback_score >= 0.10:
        feedback = "Cold... A weak semantic link."
    else:
        feedback = "Very Distant. Hardly any connection found."

    return feedback

In [47]:
def compute_game_score(similarity_score: float, relationships: list[str]) -> int:
    return int(similarity_score * 100) + len(relationships) * 50

In [48]:
import random
from IPython.display import clear_output

class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    DIM = '\033[2m'

WORD_POOL = [
    "ocean", "dog", "car", "computer", "music", 
    "forest", "book", "fire", "house", "light",
    "planet", "king", "apple", "doctor", "mountain"
] 

def print_banner():
    print(f"{Colors.OKCYAN}{'='*60}")
    print(f"{'WORDNET SEMANTIC ASSOCIATION GAME':^60}")
    print(f"{'='*60}{Colors.ENDC}\n")

def run_game():
    total_score = 0
    rounds_played = 0
    current_word = random.choice(WORD_POOL)
    last_feedback = "Enter a word related to the target!"
    last_points = 0
    last_relationships = []
    last_guess = None
    last_target = None

    clear_output()

    while True:
        print_banner()

        # Score and Status Bar
        print(f" {Colors.BOLD}Total Score:{Colors.ENDC} {Colors.OKGREEN}{total_score} pts{Colors.ENDC}  |  {Colors.BOLD}Rounds Played:{Colors.ENDC} {rounds_played}")
        print(f"{Colors.DIM}{'-'*60}{Colors.ENDC}\n")

        # Target Word and Definition
        print(f"{Colors.BOLD}Target Word:{Colors.ENDC} {Colors.WARNING}{current_word.upper()}{Colors.ENDC}")
        
        related_words = get_related_words(current_word)
        if related_words is not None and len(related_words["definitions"]) > 0:
            definition_text = related_words["definitions"][0]
            if len(definition_text) > 55:
                definition_text = definition_text[:52] + "..."
            print(f"    {Colors.DIM}Def: {definition_text}{Colors.ENDC}")
        print()

        # Last Round Results
        if last_guess is not None:
            print(f"{Colors.HEADER}┌── Last Round Results ──────────────────────────────────┐{Colors.ENDC}")
            print(f"  Last Target Word: '{Colors.BOLD}{last_target}{Colors.ENDC}'")
            print(f"  Your Word: '{Colors.BOLD}{last_guess}{Colors.ENDC}' -> {last_feedback}")
            rel_str = ', '.join(last_relationships) if last_relationships else "None"
            print(f"  Earned: {Colors.OKGREEN}+{last_points} pts{Colors.ENDC}  |  Direct Relations: {Colors.HEADER}{rel_str}{Colors.ENDC}")
            print(f"{Colors.HEADER}└────────────────────────────────────────────────────────┘{Colors.ENDC}\n")

        # Input Prompt
        print(f"Type {Colors.WARNING}'pass'{Colors.ENDC} for a new word, or {Colors.FAIL}'quit'{Colors.ENDC} to exit.")
        try:
            guess = input(f"{Colors.BOLD}Enter your related word > {Colors.ENDC}").strip().lower()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting game...")
            break

        if len(guess) == 0:
            clear_output()
            continue

        if guess == 'quit':
            print(f"\n{Colors.OKGREEN}Thanks for playing! Final Score: {total_score} pts{Colors.ENDC}")
            break

        if guess == 'pass':
            current_word = random.choice([w for w in WORD_POOL if w != current_word])
            last_guess = None
            last_feedback = "Word skipped! New target generated."
            clear_output()
            continue

        # Evaluate Guess
        similarity_score, relationships, guess_is_in_dicitonary = evaluate_association(current_word, guess)

        print(similarity_score, relationships, guess_is_in_dicitonary)

        if not guess_is_in_dicitonary:
            last_guess = guess
            last_feedback = f"{Colors.FAIL}Word not recognized in WordNet.{Colors.ENDC}"
            last_points = 0
            last_relationships = []
            clear_output()
            continue

        # Calculate Points
        last_points = compute_game_score(similarity_score, relationships)
        feedback = generate_feedback(similarity_score, relationships)

        print(last_points)

        total_score += last_points
        rounds_played += 1
        last_guess = guess
        last_target = current_word
        last_feedback = feedback
        last_relationships = relationships

        # Next Target Word
        current_word = random.choice([w for w in WORD_POOL if w != current_word])
        clear_output()

In [49]:
run_game()

             WORDNET SEMANTIC ASSOCIATION GAME              

 Total Score: 0 pts  |  Rounds Played: 0
------------------------------------------------------------

Target Word: COMPUTER
    Def: (n) a machine for performing calculations automatically

Type 'pass' for a new word, or 'quit' to exit.

Thanks for playing! Final Score: 0 pts
